# 03 — Evaluation, failure accounting, and redacted audit view

Notebook นี้อ่าน artifacts จาก notebook 02, คำนวณ metrics ใหม่จาก `predictions.jsonl` และแสดง public/redacted records. Sidecar baseline, synthetic VLM, transformed VLM และ SROIE ต้องรายงานแยก segment.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
colab_runs = sorted((PROJECT_ROOT / 'reports' / 'colab').glob('colab-*'))
if not colab_runs:
    raise FileNotFoundError('No Colab run artifacts found. Run notebook 02 first.')
RUN_DIR = colab_runs[-1]
print(RUN_DIR)

In [ ]:
import json
from IPython.display import JSON, display
from invoice_auditor.artifacts import validate_run_artifacts
from invoice_auditor.evaluation import evaluate_jsonl, read_jsonl

run_manifest = json.loads((RUN_DIR / 'run_manifest.json').read_text(encoding='utf-8'))
dataset_root = PROJECT_ROOT / 'data' / 'synthetic'
metrics = evaluate_jsonl(dataset_root / 'ground_truth.jsonl', RUN_DIR / 'predictions.jsonl')
validation = validate_run_artifacts(RUN_DIR, require_colab=False, minimum_attempts=5)
display(JSON({'run_manifest': run_manifest, 'metrics': metrics, 'validation': validation.model_dump(mode='json')}))

## Public upload-to-audit view
เลือกดู records ที่ผ่าน/ไม่ผ่านโดยไม่มี raw model response, absolute temp path, vendor name หรือ Tax ID เต็ม.

In [ ]:
predictions, parse_errors = read_jsonl(RUN_DIR / 'predictions.jsonl')
if parse_errors:
    print({'parse_errors': parse_errors})
for record in predictions:
    report = record.get('audit_report') or {}
    display(JSON({
        'source_id': record.get('image_id'),
        'status': record.get('status'),
        'raw_extraction': report.get('raw'),
        'normalized_fields': report.get('normalized'),
        'normalization_issues': report.get('normalization_issues'),
        'rules': report.get('rules'),
        'decision': report.get('decision'),
        'fallback': {
            'from': record.get('runtime', {}).get('fallback_from'),
            'reason': record.get('runtime', {}).get('fallback_reason'),
        },
        'error': {
            'stage': record.get('error_stage'),
            'type': record.get('error_type'),
            'message': record.get('error_message'),
        },
    }))